# 03 — Pull training imagery from Earth Engine

**Purpose.** Downloads the Landsat imagery used to *train* the segmentation model. For each of the ~18k
glaciers from notebook 01, it pulls every Landsat 5, 7 and 8 scene within ±50 days of the date the
glacier's GLIMS outline was drawn (`src_date`). Scenes close to that date can be paired with the outline
as a ground-truth mask (notebooks 09–10). `EePull` (`glacierview.earthengine.export`) applies
`CLOUD_COVER <= 10` and `IMAGE_QUALITY == 9` (`IMAGE_QUALITY_OLI` for L8) over each bounding box and
writes one 30 m GeoTIFF per scene, logging a line per scene so you can watch progress. Glaciers are
shuffled before pulling. A failed export for one satellite is swallowed silently
(`except Exception: pass`).

## Inputs
- **`src/glims/data/training_samples/glims_18k_bb.shp`**: bounding boxes (`bboxes`) and outline
  dates (`src_date`) from notebook 01.
- **Earth Engine account**: `ee.Authenticate()` opens a browser sign-in. Newer `earthengine-api`
  versions also need a Cloud project with Earth Engine enabled, passed as `ee.Initialize(project=...)`.
- **Existing directories**: `.../ee_landing_zone/localized_time_series_for_training_c02_t1_l2/logs/`
  must exist before `EePull` is created, and so must `landsat/` beside it. Per-glacier folders are
  created as needed.
- **Resume cell (optional)**: reads `logs/training_log_1.log`, a log from an earlier pull. This run
  logs to `log_full_training_data.log`, so on a fresh pull skip that cell or point it at the new log.
- **Environment**: the project's uv environment as the kernel, with `earthengine-api` and `geemap`.
  This is a long, network-bound run.

The last two cells (a DEM pull through `ee_helpers`, and writing `completed_batch_1.csv`) are
leftovers that reference names which no longer exist. They fail if run. DEMs are pulled in notebook 04.

## Outputs
Under `src/earth_engine/data/ee_landing_zone/localized_time_series_for_training_c02_t1_l2/`
(gitignored):
- `landsat/<glims_id>/<glims_id>_<date>_<L5|L7|L8>_C02_T1_L2_SR.tif`: one file per scene.
- `landsat/<glims_id>/meta_data/metadata_list_l{5,7,8}`: raw Earth Engine metadata, read by notebook 06.
- `logs/log_full_training_data.log`: one line per scene, read by notebook 06.

## Manual step afterwards (not in code)
The canonical copy lives in `s3://raw-training-images-t1-l2-sr` (us-west-1) as `landsat/<glims_id>/`
and `logs/`. Upload new pulls there. See `.agents/references/aws.md`.

In [1]:
import os
from datetime import datetime, timedelta

import ee
import geopandas as gpd
import numpy as np
import pandas as pd

from glacierview import config
from glacierview.earthengine import EePull  # was ee_helpers


In [3]:
# Trigger the authentication flow
ee.Authenticate()

# Initialize the library
ee.Initialize(project ="psyched-upgrade-377000")

EEException: Cannot authenticate: Invalid request.

In [ ]:
data_label = "localized_time_series_for_training_c02_t1_l2"
batch_num = 1
glacier_view_dir = str(config.REPO_ROOT)
log_dir = os.path.join(glacier_view_dir,"src","earth_engine","data","ee_landing_zone",data_label, "logs")
image_dir = os.path.join(glacier_view_dir,"src","earth_engine","data","ee_landing_zone",data_label, "landsat")

In [ ]:
ee_pull = EePull(log_dir, "full_training_data")

In [ ]:
glims_bb_path = os.path.join(glacier_view_dir,"src","glims","data","training_samples","glims_18k_bb.shp")
df = gpd.read_file(glims_bb_path)

In [ ]:
df.bboxes[0]

In [ ]:
#only use if you want to resume training

done_df = pd.read_csv(os.path.join(log_dir,"training_log_1.log"), header = None, delimiter = ';')
done_glims_ids = list(np.unique([item.split(":")[-1] for item in done_df[0]]))

df = df[~df.glac_id.isin(done_glims_ids)]

In [ ]:
df

In [ ]:
df.shape

In [ ]:
len(done_glims_ids)

In [ ]:
df.shape

In [ ]:
#shuffle the dataset
df = df.sample(frac=1).reset_index(drop=True)
n = df.shape[0]

In [ ]:
for idx,row in enumerate(df.iterrows()):
    glims_id = row[1].glac_id
    bounding_box = eval(row[1].bboxes)
    date = row[1].src_date.split("T")[0]
    date = datetime.strptime(date, '%Y-%m-%d')
    out_dir = os.path.join(image_dir,glims_id)
    if not os.path.exists(out_dir):
        os.mkdir(out_dir)

    start_date = date - timedelta(days = 50)
    end_date = date + timedelta(days = 50)

    try:
        ee_pull.export_landsat_five_images(glims_id,
                                            bounding_box,
                                            start_date,
                                            end_date,
                                            out_dir)
    except Exception:
        pass

    try:
        ee_pull.export_landsat_seven_images(glims_id,
                                            bounding_box,
                                            start_date,
                                            end_date,
                                            out_dir)
    except Exception:
        pass
    try:
        ee_pull.export_landsat_eight_images(glims_id,
                                            bounding_box,
                                            start_date,
                                            end_date,
                                            out_dir)
    except Exception:
        pass

In [ ]:
os.mkdir("data/training/dems")
for row in df.iterrows():
    glims_id = row[1].glac_id
    bounding_box = eval(row[1].bboxes)
    out_dir = os.path.join("data","training","dems")
    try:
        ee_helpers.export_nasa_dems(glims_id, bounding_box, out_dir)
    except Exception:
        pass

In [ ]:
pd.Series(completed).to_csv("completed_batch_1.csv")